# n3 · 从零构建 MLP MLP from Scratch

> **能力标签**：SH8（PyTorch 基础 / PyTorch fundamentals）

## 目标 Objectives

完成本课后，你应该能够：

1. 用 `Value` 类构建 `Neuron`（神经元）、`Layer`（层）和 `MLP`（多层感知机）。
2. 手写 **SGD**（随机梯度下降 / Stochastic Gradient Descent）训练循环。
3. 在 **XOR** 问题上训练一个两层 MLP 并观察损失下降。
4. 解释为何"这就是 PyTorch autograd 的本质"——PyTorch 的 `Tensor` 与 `value` 遵循完全相同的计算图原理。

> 对应能力：**SH8**


## 概念讲解 Concepts

### 1. 从标量到神经网络 From Scalar to Network

有了 `Value`，构建神经网络只需三个抽象层：

```
Value  →  Neuron  →  Layer  →  MLP
```

每一层都是前一层的简单封装，**梯度自动流过整个网络**——这正是 autograd 的威力。

---

### 2. 神经元 Neuron

$$\text{neuron}(x) = \tanh\left(\sum_i w_i x_i + b\right)$$

- $w_i$：权重（weight），可训练参数
- $b$：偏置（bias），可训练参数
- $\tanh$：激活函数（activation function）

```python
class Neuron:
    def __init__(self, nin):          # nin = 输入维数
        self.w = [Value(random)]      # 随机初始化权重
        self.b = Value(0.0)           # 偏置初始为 0
    def __call__(self, x):
        act = sum(wi*xi for wi,xi in zip(self.w,x)) + self.b
        return act.tanh()
    def parameters(self):
        return self.w + [self.b]
```

---

### 3. 层与 MLP Layer and MLP

`Layer` = 多个并联的 `Neuron`（每个神经元独立接收同一输入向量）。
`MLP` = 多个串联的 `Layer`（上一层输出作为下一层输入）。

---

### 4. SGD 训练循环 Training Loop

```
for step in range(steps):
    # 1. 正向传播 Forward pass
    ys_pred = [model(x) for x in xs]
    loss = sum((yp - yt)**2 for yp, yt in zip(ys_pred, ys))

    # 2. 清零梯度 Zero gradients  ← 关键！必须在 backward 前清零
    for p in model.parameters():
        p.grad = 0.0

    # 3. 反向传播 Backward pass
    loss.backward()

    # 4. 更新参数 Parameter update (SGD)
    for p in model.parameters():
        p.data -= lr * p.grad
```

**为何要清零梯度？** `grad` 使用 `+=` 累积，若不清零，历史梯度会混入当前步骤。

---

### 5. XOR 问题

XOR 是最简单的**线性不可分**（linearly non-separable）问题：

| x₁ | x₂ | y |
|----|----|---|
| 0  | 0  | −1 |
| 0  | 1  | +1 |
| 1  | 0  | +1 |
| 1  | 1  | −1 |

单层线性分类器无法解决 XOR——需要至少一个隐藏层来学习**非线性边界**。

---

### 6. 这就是 PyTorch Autograd 的本质

PyTorch 的 `torch.Tensor` 与我们的 `Value` 遵循完全相同的原理：

| 概念 | 我们的实现 | PyTorch |
|------|------|------|
| 节点 | `Value` | `Tensor` |
| 梯度 | `value.grad` | `tensor.grad` |
| 反向传播 | `value.backward()` | `tensor.backward()` |
| 参数 | `value.data` | `tensor.data` |
| 清零梯度 | `p.grad = 0` | `optimizer.zero_grad()` |
| SGD 更新 | `p.data -= lr * p.grad` | `optimizer.step()` |

理解了 `Value`，你就理解了 PyTorch。


## 示例 Worked Example

In [ ]:
# ── 完整实现：Value + Neuron + Layer + MLP ────────────────────────────────────
import math
import random


class Value:
    """标量自动微分节点（完整算子集合）。"""
    def __init__(self, data, _children=(), _op=""):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        def _backward():
            self.grad  += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        def _backward():
            self.grad  += other.data * out.grad
            other.grad += self.data  * out.grad
        out._backward = _backward
        return out

    def __pow__(self, k):
        assert isinstance(k, (int, float))
        out = Value(self.data ** k, (self,), f"**{k}")
        def _backward():
            self.grad += (k * self.data ** (k - 1)) * out.grad
        out._backward = _backward
        return out

    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other if isinstance(other, Value) else Value(-other))
    def __truediv__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return self * other ** -1

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), "tanh")
        def _backward(): self.grad += (1 - t * t) * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0.0, self.data), (self,), "relu")
        def _backward(): self.grad += (1.0 if self.data > 0 else 0.0) * out.grad
        out._backward = _backward
        return out

    def exp(self):
        out = Value(math.exp(self.data), (self,), "exp")
        def _backward(): self.grad += out.data * out.grad
        out._backward = _backward
        return out

    __radd__ = __add__
    __rmul__ = __mul__

    def backward(self):
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev: build(child)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for v in reversed(topo): v._backward()

    def __repr__(self):
        return f"Value({self.data:.4f})"


class Neuron:
    """单个神经元 (Single neuron): out = tanh(w·x + b)."""
    def __init__(self, nin, seed=0):
        rng = random.Random(seed)
        self.w = [Value(rng.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(0.0)

    def __call__(self, x):
        # 线性组合 + 激活函数
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh()

    def parameters(self):
        return self.w + [self.b]


class Layer:
    """全连接层 (Fully-connected layer): nout 个并联神经元."""
    def __init__(self, nin, nout, seed=0):
        self.neurons = [Neuron(nin, seed + i) for i in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]


class MLP:
    """多层感知机 (Multi-Layer Perceptron)."""
    def __init__(self, nin, nouts, seed=0):
        sizes = [nin] + nouts
        self.layers = [Layer(sizes[i], sizes[i+1], seed + 10*i)
                       for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]


# ── 检查模型结构 ───────────────────────────────────────────────────────────
model = MLP(2, [4, 1], seed=1)   # 2 输入, 隐层 4 神经元, 1 输出
n_params = len(model.parameters())
print(f"MLP(2, [4, 1]) 参数总数: {n_params}")
print(f"  期望: 2 输入层 → 隐层 4×(2+1)=12 参数, 隐层 → 输出 1×(4+1)=5 参数")
print(f"  合计: {12 + 5} 参数")
assert n_params == 17, f"参数数量错误: {n_params}"
print("✓ 模型结构正确")


In [ ]:
# ── 在 XOR 上训练 MLP ─────────────────────────────────────────────────────
# XOR 数据集（用 ±1 标签方便 MSE 损失）
XOR_DATA = [
    ([0.0, 0.0], -1.0),
    ([0.0, 1.0],  1.0),
    ([1.0, 0.0],  1.0),
    ([1.0, 1.0], -1.0),
]

model = MLP(2, [4, 1], seed=1)

# SGD 超参数
LR    = 0.1
STEPS = 300

history = []
for step in range(STEPS):
    # 1. 正向传播：计算 MSE 损失
    loss = sum(
        ((model(x) - y) ** 2 for x, y in XOR_DATA),
        Value(0.0)
    )

    # 2. 清零梯度（必须在 backward 前！）
    for p in model.parameters():
        p.grad = 0.0

    # 3. 反向传播
    loss.backward()

    # 4. SGD 更新
    for p in model.parameters():
        p.data -= LR * p.grad

    history.append(loss.data)
    if step % 50 == 0 or step == STEPS - 1:
        print(f"  step {step:3d}  loss = {loss.data:.6f}")

print()
print(f"最终损失 Final loss: {history[-1]:.6f}  (目标 < 0.1)")
assert history[-1] < 0.1, f"训练未收敛: loss={history[-1]:.4f}"
print("✓ XOR 训练成功收敛")


In [ ]:
# ── 验证预测结果 ──────────────────────────────────────────────────────────
print("XOR 预测验证:")
print(f"  {'输入 x':>12}  {'真实 y':>8}  {'预测':>10}  {'符号':>6}")
print("  " + "-" * 42)
for x, y in XOR_DATA:
    pred = model(x).data
    sign = "+" if pred > 0 else "-"
    ok = (pred > 0) == (y > 0)
    status = "✓" if ok else "✗"
    print(f"  {str(x):>12}  {y:>8.0f}  {pred:>10.4f}  {sign:>6}  {status}")

print()
print("这就是 PyTorch autograd 的本质：")
print("  torch.Tensor  ≈  Value（支持整个运算图和梯度传播）")
print("  optimizer.zero_grad()  ≈  for p in params: p.grad = 0.0")
print("  loss.backward()  ≈  loss.backward()（完全相同的调用！）")
print("  optimizer.step()  ≈  for p in params: p.data -= lr * p.grad")


In [ ]:
# ── 有限差分验证：网络梯度也是正确的 ────────────────────────────────────────

def compute_loss(model):
    """重新计算 XOR MSE 损失."""
    return sum(
        ((model(x) - y) ** 2 for x, y in XOR_DATA),
        Value(0.0)
    )

# 取第一个参数验证梯度
p0 = model.parameters()[0]

# Autograd 梯度
for p in model.parameters(): p.grad = 0.0
loss = compute_loss(model)
loss.backward()
ag = p0.grad

# 数值梯度（有限差分）
h = 1e-4
p0.data += h
loss_plus = compute_loss(model).data
p0.data -= 2 * h
loss_minus = compute_loss(model).data
p0.data += h   # 恢复原值
nd = (loss_plus - loss_minus) / (2 * h)

print(f"参数 p0 = {p0.data:.4f}")
print(f"  autograd  梯度: {ag:.6f}")
print(f"  有限差分  梯度: {nd:.6f}")
print(f"  误差: {abs(ag - nd):.2e}")
assert abs(ag - nd) < 1e-3, "梯度不匹配"
print("✓ 网络梯度验证通过")


## 动手 Exercise

In [ ]:
# ── 动手练习：调整学习率，观察收敛速度 ──────────────────────────────────────
# 实验：用相同的模型初始化，对比 lr=0.01, 0.1, 0.5 三种学习率
# 观察：各学习率在 200 步后的损失；哪个收敛最快？哪个不稳定？

import math
import random


class Value:
    def __init__(self, data, _children=(), _op=""):
        self.data = float(data); self.grad = 0.0
        self._backward = lambda: None; self._prev = set(_children); self._op = _op
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data+other.data, (self,other), "+")
        def _bw(): self.grad+=out.grad; other.grad+=out.grad
        out._backward=_bw; return out
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data*other.data, (self,other), "*")
        def _bw(): self.grad+=other.data*out.grad; other.grad+=self.data*out.grad
        out._backward=_bw; return out
    def __pow__(self, k):
        out = Value(self.data**k, (self,), f"**{k}")
        def _bw(): self.grad += k*self.data**(k-1)*out.grad
        out._backward=_bw; return out
    def __neg__(self): return self * -1
    def __sub__(self, other): return self+(-other if isinstance(other,Value) else Value(-other))
    def tanh(self):
        t=math.tanh(self.data); out=Value(t,(self,),"tanh")
        def _bw(): self.grad+=(1-t*t)*out.grad
        out._backward=_bw; return out
    __radd__=__add__; __rmul__=__mul__
    def backward(self):
        topo,visited=[],set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for c in v._prev: build(c)
                topo.append(v)
        build(self); self.grad=1.0
        for v in reversed(topo): v._backward()


class Neuron:
    def __init__(self, nin, seed=0):
        rng=random.Random(seed)
        self.w=[Value(rng.uniform(-1,1)) for _ in range(nin)]; self.b=Value(0.0)
    def __call__(self, x):
        return sum((wi*xi for wi,xi in zip(self.w,x)), self.b).tanh()
    def parameters(self): return self.w+[self.b]

class Layer:
    def __init__(self, nin, nout, seed=0):
        self.neurons=[Neuron(nin,seed+i) for i in range(nout)]
    def __call__(self, x):
        outs=[n(x) for n in self.neurons]; return outs[0] if len(outs)==1 else outs
    def parameters(self): return [p for n in self.neurons for p in n.parameters()]

class MLP:
    def __init__(self, nin, nouts, seed=0):
        sizes=[nin]+nouts
        self.layers=[Layer(sizes[i],sizes[i+1],seed+10*i) for i in range(len(nouts))]
    def __call__(self, x):
        for l in self.layers: x=l(x)
        return x
    def parameters(self): return [p for l in self.layers for p in l.parameters()]


XOR_DATA = [([0.0,0.0],-1.0),([0.0,1.0],1.0),([1.0,0.0],1.0),([1.0,1.0],-1.0)]
STEPS = 200

print(f"{'lr':>6}  {'loss @ step 0':>14}  {'loss @ step 200':>15}  {'收敛?':>6}")
print("-" * 50)

for lr in [0.01, 0.1, 0.5]:
    m = MLP(2, [4, 1], seed=1)   # 相同初始化
    for step in range(STEPS):
        loss = sum(((m(x)-y)**2 for x,y in XOR_DATA), Value(0.0))
        for p in m.parameters(): p.grad = 0.0
        loss.backward()
        for p in m.parameters(): p.data -= lr * p.grad
    final = loss.data
    converged = "✓" if final < 0.5 else "✗"
    print(f"{lr:>6.2f}  {'(see first step)':>14}  {final:>15.6f}  {converged:>6}")

print()
print("观察：lr 太小收敛慢；lr 适中收敛好；lr 太大可能发散或震荡。")


## 延伸阅读 Further Reading

1. **Karpathy «makemore» + «nanoGPT»**：<https://github.com/karpathy> — 从 bigram 到 GPT，同样的 autograd 思路扩展到语言模型。
2. **PyTorch «Writing Custom Autograd Functions»**：<https://pytorch.org/docs/stable/notes/extending.html> — 如何在 PyTorch 中添加自定义算子，与本课 `_backward` 完全类比。
3. **Goodfellow et al. «Deep Learning»** 第 6 章（前馈网络）和第 8 章（SGD 优化）——理论基础。
4. **Sebastian Ruder «Overview of Gradient Descent Optimization Algorithms»**：<https://ruder.io/optimizing-gradient-descent/> — 从 SGD 到 Adam 的全面综述。
5. **Universal Approximation Theorem**：两层 MLP 能近似任意连续函数——XOR 只是最简单的一步。


## 关联练习 Related Assignments

```bash
python check.py nanograd-l3
```

> 实现 `Neuron`、`Layer`、`MLP` 类，并完成 `train_xor` 函数，在 XOR 数据上训练至损失低于阈值。

> 能力标签：**SH8** · threshold ≥ 0.7
